# AI-Based Retail Inventory Intelligence System

## Notebook 2 : Deployment & Prediction Pipeline

### Objective

This notebook demonstrates how the trained machine learning model is integrated into a real-world Retail Inventory Intelligence System.

The notebook includes:

- Loading the trained model
- Loading label encoders
- Loading feature columns
- Creating a product database
- Building the prediction pipeline
- Inventory intelligence engine
- Testing the complete system


## Step 1 : Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 : Import Required Libraries

In [2]:
import pandas as pd
import numpy as np
import joblib

## Step 3 : Load Saved AI Model

In [3]:
# Load trained model
model = joblib.load(
    "/content/drive/MyDrive/Retail_Inventory_Project/Models/final_inventory_model.pkl"
)

# Load encoders
encoders = joblib.load(
    "/content/drive/MyDrive/Retail_Inventory_Project/Models/label_encoders.pkl"
)

# Load feature order
feature_columns = joblib.load(
    "/content/drive/MyDrive/Retail_Inventory_Project/Models/feature_columns.pkl"
)

print("AI System Loaded Successfully!")

AI System Loaded Successfully!


In [4]:
print(type(model))

print()

print(encoders.keys())

print()

print(feature_columns)

<class 'sklearn.ensemble._forest.RandomForestRegressor'>

dict_keys(['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality'])

['Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality', 'Month', 'Day']


In [5]:
## Step 5 : Load Product Database

In [6]:
# Load original dataset

dataset_path = "/content/drive/MyDrive/Retail_Inventory_Project/Dataset/archive.zip"

products_df = pd.read_csv(
    dataset_path,
    compression="zip"
)

print("Product database loaded successfully!")
print(products_df.shape)

products_df.head()

Product database loaded successfully!
(73100, 15)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


## Step 6 : Create Product Master Database

In [7]:
# Select only required product information

product_master = products_df[
    [
        "Store ID",
        "Product ID",
        "Category",
        "Region",
        "Price",
        "Competitor Pricing",
        "Seasonality"
    ]
]

# Remove duplicate products

product_master = product_master.drop_duplicates()

print("Product Master Database Created!")

print("\nShape :", product_master.shape)

product_master.head()

Product Master Database Created!

Shape : (73100, 7)


,Store ID,Product ID,Category,Region,Price,Competitor Pricing,Seasonality
0,S001,P0001,Groceries,North,33.50,29.69,Autumn
1,S001,P0002,Toys,South,63.01,66.16,Autumn
2,S001,P0003,Toys,West,27.99,31.32,Summer
3,S001,P0004,Toys,North,32.72,34.74,Autumn
4,S001,P0005,Electronics,East,73.64,68.95,Summer


## Step 7 : Create Latest Product Master Database

In [8]:
# Step 7 : Create Latest Product Database

# Find the latest date in the dataset
latest_date = products_df["Date"].max()

print("Latest Date :", latest_date)

Latest Date : 2024-01-01


In [9]:
# Keep only the latest records

latest_products = products_df[
    products_df["Date"] == latest_date
].copy()

print("Latest Product Database Created!")
print("Shape :", latest_products.shape)

latest_products.head(10)

Latest Product Database Created!
Shape : (100, 15)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
73000,2024-01-01,S001,P0001,Toys,East,223,40,93,53.56,55.26,15,Rainy,0,59.37,Winter
73001,2024-01-01,S001,P0002,Furniture,West,217,99,73,101.47,44.84,10,Snowy,0,41.09,Winter
73002,2024-01-01,S001,P0003,Toys,East,69,64,191,76.85,21.94,5,Snowy,0,25.07,Spring
73003,2024-01-01,S001,P0004,Groceries,West,338,182,152,192.43,86.37,0,Cloudy,1,82.98,Spring
73004,2024-01-01,S001,P0005,Groceries,North,471,272,167,266.62,15.58,20,Snowy,0,20.04,Autumn
73005,2024-01-01,S001,P0006,Electronics,North,305,87,114,91.48,64.76,10,Snowy,0,66.83,Summer
73006,2024-01-01,S001,P0007,Clothing,East,256,9,184,25.06,77.44,10,Cloudy,1,74.20,Spring
73007,2024-01-01,S001,P0008,Electronics,East,315,233,108,226.38,31.34,0,Cloudy,1,27.26,Summer
73008,2024-01-01,S001,P0009,Electronics,South,167,22,188,28.21,96.80,15,Snowy,0,95.06,Spring
73009,2024-01-01,S001,P0010,Furniture,West,167,70,107,86.62,40.95,10,Sunny,0,36.46,Winter


Step 8:Create the function

In [10]:
# Step 8 : Search Product

def get_product_details(product_id):

    product = latest_products[
        latest_products["Product ID"] == product_id
    ]

    if product.empty:
        print("❌ Product not found!")
        return None

    return product

In [11]:
get_product_details("P0005")

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
73004,2024-01-01,S001,P0005,Groceries,North,471,272,167,266.62,15.58,20,Snowy,0,20.04,Autumn
73024,2024-01-01,S002,P0005,Toys,West,464,200,44,198.35,16.98,5,Snowy,1,18.39,Spring
73044,2024-01-01,S003,P0005,Electronics,West,411,291,200,298.09,82.16,20,Sunny,1,79.32,Summer
73064,2024-01-01,S004,P0005,Toys,East,217,112,88,103.44,77.66,0,Cloudy,1,82.37,Autumn
73084,2024-01-01,S005,P0005,Toys,South,372,154,143,150.53,96.72,0,Cloudy,1,97.84,Winter


step 9.1:Prepare Product for Prediction

In [24]:
# Step 9.1 : Prepare Product for Prediction

def prepare_input(product_row):

    input_data = product_row.copy()

    # Extract Year, Month and Day from Date
    input_data["Date"] = pd.to_datetime(input_data["Date"])

    input_data["Year"] = input_data["Date"].dt.year
    input_data["Month"] = input_data["Date"].dt.month
    input_data["Day"] = input_data["Date"].dt.day

    # Remove columns not used by the model
    input_data = input_data.drop(
    columns=[
        "Date",
        "Units Sold",
        "Units Ordered",
        "Demand Forecast",
        "Year"
    ],
    errors="ignore"
)
    return input_data

In [25]:
product = get_product_details("P0005")

prepared = prepare_input(product)

prepared.head()

,Store ID,Product ID,Category,Region,Inventory Level,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Month,Day
73004,S001,P0005,Groceries,North,471,15.58,20,Snowy,0,20.04,Autumn,1,1
73024,S002,P0005,Toys,West,464,16.98,5,Snowy,1,18.39,Spring,1,1
73044,S003,P0005,Electronics,West,411,82.16,20,Sunny,1,79.32,Summer,1,1
73064,S004,P0005,Toys,East,217,77.66,0,Cloudy,1,82.37,Autumn,1,1
73084,S005,P0005,Toys,South,372,96.72,0,Cloudy,1,97.84,Winter,1,1


Step 9.2: Encode the Categorical Columns

In [14]:
# Step 9.2 : Encode categorical columns

def encode_input(input_data):

    encoded = input_data.copy()

    for col in encoders.keys():

        encoded[col] = encoders[col].transform(encoded[col])

    return encoded

In [26]:
encoded = encode_input(prepared)

encoded.head()

,Store ID,Product ID,Category,Region,Inventory Level,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Month,Day
73004,0,4,3,1,471,15.58,20,2,0,20.04,0,1,1
73024,1,4,4,3,464,16.98,5,2,1,18.39,1,1,1
73044,2,4,1,3,411,82.16,20,3,1,79.32,2,1,1
73064,3,4,4,0,217,77.66,0,0,1,82.37,0,1,1
73084,4,4,4,2,372,96.72,0,0,1,97.84,3,1,1


In [27]:
encoded = encode_input(prepared)

encoded.head()

,Store ID,Product ID,Category,Region,Inventory Level,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Month,Day
73004,0,4,3,1,471,15.58,20,2,0,20.04,0,1,1
73024,1,4,4,3,464,16.98,5,2,1,18.39,1,1,1
73044,2,4,1,3,411,82.16,20,3,1,79.32,2,1,1
73064,3,4,4,0,217,77.66,0,0,1,82.37,0,1,1
73084,4,4,4,2,372,96.72,0,0,1,97.84,3,1,1


In [28]:
# Arrange columns exactly as used during training
encoded = encoded[feature_columns]

print(encoded.columns.tolist())

['Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality', 'Month', 'Day']


In [18]:
print(encoded.columns.tolist())

['Store ID', 'Product ID', 'Category', 'Region', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality', 'Year', 'Month', 'Day']


In [19]:
print(feature_columns)

['Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality', 'Month', 'Day']


In [29]:
prediction = model.predict(encoded)

print("Predicted Units Sold:")
print(prediction)

Predicted Units Sold:
[231.94021037 224.75034071 216.10009044 110.82496846 210.63922106]


In [30]:
result = prepared.copy()

result["Predicted Units Sold"] = prediction

result

,Store ID,Product ID,Category,Region,Inventory Level,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Month,Day,Predicted Units Sold
73004,S001,P0005,Groceries,North,471,15.58,20,Snowy,0,20.04,Autumn,1,1,231.940210
73024,S002,P0005,Toys,West,464,16.98,5,Snowy,1,18.39,Spring,1,1,224.750341
73044,S003,P0005,Electronics,West,411,82.16,20,Sunny,1,79.32,Summer,1,1,216.100090
73064,S004,P0005,Toys,East,217,77.66,0,Cloudy,1,82.37,Autumn,1,1,110.824968
73084,S005,P0005,Toys,South,372,96.72,0,Cloudy,1,97.84,Winter,1,1,210.639221


In [33]:
result[["Product ID", "Predicted Units Sold"]]

,Product ID,Predicted Units Sold
73004,P0005,231.940210
73024,P0005,224.750341
73044,P0005,216.100090
73064,P0005,110.824968
73084,P0005,210.639221
